# ModelEdge — QLoRA Fine-tuning on Kaggle 2x T4

**Setup in Kaggle:**
1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Secrets → add `WANDB_API_KEY` with your W&B key
3. Run all cells top-to-bottom

Adapter weights are saved to `/kaggle/working/outputs/finetuned`.

In [ ]:
# Verify both GPUs are visible
!nvidia-smi

In [ ]:
!pip install -q unsloth[colab-new] peft accelerate datasets bitsandbytes trl wandb
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'

In [ ]:
# Load W&B key from Kaggle secrets (add it in Settings → Secrets)
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')

import wandb
wandb.login()

In [ ]:
OUTPUT_DIR = '/kaggle/working/outputs/finetuned'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output dir: {OUTPUT_DIR}')

In [ ]:
# Clone the repo
!git clone https://github.com/sakshiasati17/ModelEdge.git /kaggle/working/ModelEdge
%cd /kaggle/working/ModelEdge

In [ ]:
# Prepare dataset
!python data/prepare_dataset.py --dataset medqa --output data/processed/

In [ ]:
# Inspect a few samples
import json

with open('data/processed/medqa/train.jsonl') as f:
    for i, line in enumerate(f):
        rec = json.loads(line)
        print(f'--- Sample {i+1} ---')
        print(rec['text'][:400])
        print()
        if i >= 2:
            break

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048
MODEL_NAME = 'unsloth/Llama-3.2-3B-Instruct'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
print('Params:', sum(p.numel() for p in model.parameters()) / 1e6, 'M')

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
model.print_trainable_parameters()

In [ ]:
from datasets import Dataset

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

train_ds = Dataset.from_list(load_jsonl('data/processed/medqa/train.jsonl'))
val_ds   = Dataset.from_list(load_jsonl('data/processed/medqa/val.jsonl'))
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

In [ ]:
import torch
from accelerate import Accelerator
from trl import SFTTrainer
from transformers import TrainingArguments

accelerator = Accelerator()
print(f'Accelerator: {accelerator.num_processes} process(es) on {accelerator.device}')

wandb.init(project='modeledge-medqa', name='qlora-llama3.2-3b-kaggle-2xT4')

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    optim='adamw_8bit',
    fp16=True,
    logging_steps=10,
    evaluation_strategy='steps',
    eval_steps=100,
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to='wandb',
    seed=42,
    ddp_find_unused_parameters=accelerator.num_processes > 1,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    args=training_args,
)

In [ ]:
trainer.train()
wandb.finish()

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Adapter saved → {OUTPUT_DIR}')

In [ ]:
# Quick inference test
FastLanguageModel.for_inference(model)

test_prompt = (
    'Below is a medical question. Answer it accurately and concisely.\n\n'
    '### Instruction:\nWhat is the first-line treatment for hypertension?\n\n'
    '### Input:\n\n'
    '### Response:\n'
)

inputs = tokenizer(test_prompt, return_tensors='pt').to('cuda')
outputs = model.generate(**inputs, max_new_tokens=128, temperature=0.1, do_sample=True)
generated = outputs[0][inputs['input_ids'].shape[1]:]
print(tokenizer.decode(generated, skip_special_tokens=True))

In [ ]:
# vLLM serve with both T4s (run in a terminal tab inside Kaggle)
print('To serve with both T4s, run in a terminal:')
print()
print('  python -m vllm.entrypoints.openai.api_server \\')
print(f'    --model {OUTPUT_DIR} \\')
print('    --tensor-parallel-size 2 \\')
print('    --host 0.0.0.0 --port 8001')